# ChpWiseDB — LLM Answer Generation Pipeline

- Embedding:BGE-large (BAAI/bge-large-en-v1.5)
- Reranker 1: CrossEncoder (cross-encoder/ms-marco-MiniLM-L-6-v2) 
- Reranker 2: Qwen3-Reranker-4B 
- LLM: Qwen2.5-7B-Instruct 


cross_norm_score: sigmoid of CrossEncoder logit → absolute 0–1 relevance probability  
qwen_norm_score: yes/no log-softmax probability → absolute 0–1 relevance probability  




## 1. Imports & Config

In [ ]:
import os
import re
import time
import numpy as np
import pandas as pd
import torch
import transformers

from collections import defaultdict, Counter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Paths ─────────────────────────────────────────────────────────────────
JCI_PDF     = '/Users/siri/Desktop/sghpolicyreview/data/ESS Standards_7th Ed .pdf'
DB_SAVE_DIR = 'saved_dbs'

# ── Embedding model ───────────────────────────────────────────────────────
EMBEDDING_MODEL_NAME   = 'BGE-large'
EMBEDDING_MODEL_STRING = 'BAAI/bge-large-en-v1.5'

# ── Reranker models ───────────────────────────────────────────────────────
CROSS_MODEL_ID   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
QWEN_MODEL_ID = 'Qwen/Qwen3-Reranker-4B'

# ── LLM ───────────────────────────────────────────────────────────────────
LLM_MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

# ── Retrieval settings ────────────────────────────────────────────────────
PER_DB_K     = 8   # chunks fetched per chapter DB
FINAL_K      = 10  # chunks kept after global re-sort and dedup

# ── LLM settings ─────────────────────────────────────────────────────────
LLM_TOP_K      = 5    # top reranked chunks, each generates its own answer
LLM_MAX_TOKENS = 512

# ── JCI chapter codes ─────────────────────────────────────────────────────
VALID_CODES = {
    'APR', 'IPSG', 'ACC', 'PCC', 'AOP', 'COP', 'ASC', 'MMU',
    'QPS', 'PCI', 'GLD', 'FMS', 'SQE', 'MOI', 'MPE', 'HRP',
}
END_OF_BOOK_MARKERS = ['glossary', 'index', 'appendix', 'references']

print('Config loaded.')

## 2. Load PDF

In [ ]:
def load_pdf(path: str):
    pages = PyPDFLoader(path).load()
    print(f'Loaded {len(pages)} pages.')
    return pages

pages = load_pdf(JCI_PDF)

## 3. Group Pages by Chapter & Build Chunks

In [ ]:
def detect_chapter(text: str):
    """Returns the JCI chapter code if this page is a chapter title page."""
    for line in text.split('\n'):
        m = re.match(r'^.+\(([A-Z]{3,4})\)\s*$', line.strip())
        if m and m.group(1) in VALID_CODES:
            return m.group(1)
    return None


def is_end_of_book(text: str) -> bool:
    """Returns True if this page looks like back-matter (glossary, index etc)."""
    return any(marker in text.strip()[:200].lower() for marker in END_OF_BOOK_MARKERS)


def group_pages_by_chapter(pages) -> dict:
    """Assigns each PDF page to its chapter based on detected chapter headings."""
    chapter_pages = defaultdict(list)
    current = 'UNKNOWN'
    for page in pages:
        detected = detect_chapter(page.page_content)
        if detected:
            current = detected
        chapter_pages[current].append(page)

    if 'HRP' in chapter_pages:
        before = len(chapter_pages['HRP'])
        chapter_pages['HRP'] = [
            p for p in chapter_pages['HRP'] if not is_end_of_book(p.page_content)
        ]
        print(f'HRP: removed {before - len(chapter_pages["HRP"])} end-of-book pages.')

    print('\n=== Chapter page counts ===')
    for ch, pgs in chapter_pages.items():
        print(f'  {ch}: {len(pgs)} pages')
    return chapter_pages


def build_chapter_chunks(chapter_pages: dict, chunk_size=1000, chunk_overlap=250) -> dict:
    """Splits each chapter into LangChain Document chunks, one per standard where possible."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '. ', ' '],
    )
    chapter_chunks = {}

    for chapter, pg_list in chapter_pages.items():
        full_text = '\n'.join(p.page_content for p in pg_list)
        full_text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', full_text)
        full_text = re.sub(r'\n{3,}', '\n\n', full_text)

        toc = re.search(
            r'Standards,\s+Intents,\s+and\s+Measurable\s+Elements',
            full_text, re.IGNORECASE
        )
        if toc:
            full_text = full_text[toc.start():]

        pattern = rf'(?:\n|^)(?:Standard|Requirement)\s+({chapter}\s*\.\s*\d+(?:\s*\.\s*\d+)?(?:\s*\.\s*\d+)?)\b'
        splits  = re.split(pattern, full_text)

        docs = []
        if splits[0].strip():
            docs.append(Document(
                page_content=splits[0].strip(),
                metadata={'chapter': chapter, 'standard_code': 'INTRO'},
            ))

        for i in range(1, len(splits), 2):
            if i + 1 >= len(splits):
                break
            code    = splits[i].strip().replace(' ', '')
            content = f'{code}\n{splits[i + 1]}'

            if len(content.strip()) < 100:
                continue
            if len(content) <= chunk_size:
                docs.append(Document(
                    page_content=content.strip(),
                    metadata={'chapter': chapter, 'standard_code': code},
                ))
            else:
                for j, sub in enumerate(splitter.split_text(content)):
                    if len(sub.strip()) < 100:
                        continue
                    docs.append(Document(
                        page_content=sub.strip(),
                        metadata={'chapter': chapter, 'standard_code': code, 'chunk_index': j},
                    ))

        chapter_chunks[chapter] = docs
        unique = len(set(d.metadata['standard_code'] for d in docs))
        print(f'  {chapter}: {len(docs)} chunks across {unique} standards')

    return chapter_chunks


chapter_pages  = group_pages_by_chapter(pages)
chapter_chunks = build_chapter_chunks(chapter_pages)

## 4. Gold Standard Queries

In [ ]:
gold_standard = [
    # ACC.1 — Screening and Admission
    {'query': 'What screening criteria should be used when a patient arrives at the emergency department?', 'expected_standard': 'ACC.1',     'expected_chapter': 'ACC'},
    {'query': 'How should hospitals handle patients with communicable diseases during screening?',           'expected_standard': 'ACC.1',     'expected_chapter': 'ACC'},
    # ACC.1.1 — Delays in Care
    {'query': 'How should hospitals inform patients about delays in diagnostic or treatment services?',      'expected_standard': 'ACC.1.1',   'expected_chapter': 'ACC'},
    # ACC.2 — Patient Flow
    {'query': 'How should hospitals manage patient flow to prevent emergency department overcrowding?',      'expected_standard': 'ACC.2',     'expected_chapter': 'ACC'},
    {'query': 'What are the requirements for managing patients waiting for inpatient beds in the ED?',       'expected_standard': 'ACC.2',     'expected_chapter': 'ACC'},
    # ACC.2.2 — Admission Information
    {'query': 'What information must be provided to patients when they are admitted to the hospital?',       'expected_standard': 'ACC.2.2',   'expected_chapter': 'ACC'},
    # ACC.2.3 — ICU / Specialised Services
    {'query': 'What criteria should be established for admission to intensive care units?',                  'expected_standard': 'ACC.2.3',   'expected_chapter': 'ACC'},
    {'query': 'How should hospitals determine when patients no longer need intensive care?',                 'expected_standard': 'ACC.2.3',   'expected_chapter': 'ACC'},
    {'query': 'What are the requirements for psychiatric patient admission criteria?',                       'expected_standard': 'ACC.2.3',   'expected_chapter': 'ACC'},
    {'query': 'What criteria should hospitals use for admission and discharge from intensive or specialized care departments?', 'expected_standard': 'ACC.2.3', 'expected_chapter': 'ACC'},
    # ACC.3.1 — Continuity of Care
    {'query': "Who should be responsible for coordinating a patient's care throughout their hospital stay?", 'expected_standard': 'ACC.3.1',  'expected_chapter': 'ACC'},
    {'query': 'What processes should be in place to ensure continuity when care responsibility is transferred?', 'expected_standard': 'ACC.3.1', 'expected_chapter': 'ACC'},
    # ACC.4 — Discharge Planning
    {'query': 'When should discharge planning begin for hospitalized patients?',                            'expected_standard': 'ACC.4',     'expected_chapter': 'ACC'},
    # ACC.4.1 — Discharge Education
    {'query': 'What education must be provided to patients and families before discharge?',                  'expected_standard': 'ACC.4.1',   'expected_chapter': 'ACC'},
    {'query': 'What education and follow-up is required for a complex care patient being discharged home?',  'expected_standard': 'ACC.4.1',   'expected_chapter': 'ACC'},
    # ACC.4.2 — Discharge Summary
    {'query': "What information must be included in a patient's discharge summary?",                        'expected_standard': 'ACC.4.2',   'expected_chapter': 'ACC'},
    # ACC.4.2.1 — ED Documentation
    {'query': 'What specific information must be documented for emergency department patients?',             'expected_standard': 'ACC.4.2.1', 'expected_chapter': 'ACC'},
    # ACC.4.3 — Complex Care
    {'query': 'How should hospitals manage information for patients with multiple complex diagnoses?',       'expected_standard': 'ACC.4.3',   'expected_chapter': 'ACC'},
    # ACC.4.4 — Against Medical Advice
    {'query': 'What procedures should be followed when a patient wants to leave against medical advice?',   'expected_standard': 'ACC.4.4',   'expected_chapter': 'ACC'},
    {'query': 'What documentation is required when a patient leaves without notifying staff?',              'expected_standard': 'ACC.4.4.1', 'expected_chapter': 'ACC'},
    # ACC.5 — Transfer
    {'query': 'What criteria should be used to determine if a patient needs to be transferred to another facility?', 'expected_standard': 'ACC.5', 'expected_chapter': 'ACC'},
    # ACC.5.1 — Transfer Documentation
    {'query': 'What information must be provided to the receiving hospital when transferring a patient?',    'expected_standard': 'ACC.5.1',   'expected_chapter': 'ACC'},
    {'query': 'What are all the documentation requirements when a patient is transferred from the ICU?',     'expected_standard': 'ACC.5.1',   'expected_chapter': 'ACC'},
    # ACC.6 — Transportation
    {'query': "What are the hospital's responsibilities for patient transportation services?",               'expected_standard': 'ACC.6',     'expected_chapter': 'ACC'},
    {'query': 'What infection control measures should be considered during patient transportation?',         'expected_standard': 'ACC.6',     'expected_chapter': 'ACC'},
]

print(f'Gold standard: {len(gold_standard)} queries')
for std, n in sorted(Counter(q['expected_standard'] for q in gold_standard).items()):
    print(f'  {std}: {n}')

## 5. Build & Save FAISS Database
> **Run once only.** Skips if the BGE-large DB already exists.

In [ ]:
os.makedirs(DB_SAVE_DIR, exist_ok=True)

model_dir = os.path.join(DB_SAVE_DIR, EMBEDDING_MODEL_NAME)

if os.path.exists(model_dir):
    print(f'[{EMBEDDING_MODEL_NAME}] DB already exists — skipping.')
else:
    print(f'[{EMBEDDING_MODEL_NAME}] Building DBs...')
    os.makedirs(model_dir, exist_ok=True)

    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_STRING,
        encode_kwargs={'normalize_embeddings': True},
    )
    for chapter, docs in chapter_chunks.items():
        if docs:
            db = FAISS.from_documents(docs, embeddings)
            db.save_local(os.path.join(model_dir, chapter))
            print(f'  Saved: {chapter}')

    print(f'[{EMBEDDING_MODEL_NAME}] Done.')

## 6. Retrieval & Score Normalisation

In [ ]:
def retrieve(query: str, dbs: dict) -> list:
    """
    Query every chapter DB, sort globally by FAISS L2 distance
    (lower = more similar), deduplicate, return top FINAL_K chunks.
    Returns: list of (Document, raw_l2_score)
    """
    pool = []
    for chapter, db in dbs.items():
        for doc, score in db.similarity_search_with_score(query, k=PER_DB_K):
            doc.metadata['chapter'] = chapter
            pool.append((doc, score))

    pool.sort(key=lambda x: x[1])  # ascending: lowest L2 = most similar

    seen, unique = set(), []
    for doc, score in pool:
        key = (doc.page_content, doc.metadata.get('standard_code'))
        if key not in seen:
            unique.append((doc, score))
            seen.add(key)

    return unique[:FINAL_K]


def sigmoid(x: float) -> float:
    """Converts a CrossEncoder logit to a 0-1 relevance probability."""
    return round(1.0 / (1.0 + np.exp(-x)), 4)


print('Retrieval functions ready.')

## 7. Load Rerankers

In [ ]:
# ── CrossEncoder ──────────────────────────────────────────────────────────
# Returns list of (doc, norm_score) sorted by score descending.
# norm_score = sigmoid(logit) — absolute 0-1 relevance probability.

print(f'Loading Cross reranker: {CROSS_MODEL_ID}')

cross_model = CrossEncoder(CROSS_MODEL_ID)

def rerank_cross(query: str, retrieved: list) -> list:
    """
    Returns list of (doc, norm_score) sorted by norm_score descending.
    norm_score = sigmoid(raw_logit) — absolute 0-1 relevance probability.
    """
    docs       = [doc for doc, _ in retrieved]
    pairs      = [(query, doc.page_content) for doc in docs]
    raw_scores = cross_model.predict(pairs).tolist()
    combined   = [
        (doc, sigmoid(raw))
        for doc, raw in zip(docs, raw_scores)
    ]
    return sorted(combined, key=lambda x: x[1], reverse=True)

print(f'CrossEncoder loaded: {CROSS_MODEL_ID}')

In [ ]:
# ── Qwen Reranker ─────────────────────────────────────────────────────────
# Returns list of (doc, norm_score) sorted by score descending.
# norm_score = yes/no log-softmax probability — absolute 0-1.
# A score near 1.0 means the model is near-certain the chunk is relevant.
# Check qwen_retrieved_standard == expected_standard to verify.

print(f'Loading Qwen reranker: {QWEN_MODEL_ID}')

qwen_r_tok = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, padding_side='left')
qwen_r_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID, device_map='auto'
).eval()

qwen_r_true_id  = qwen_r_tok.convert_tokens_to_ids('yes')
qwen_r_false_id = qwen_r_tok.convert_tokens_to_ids('no')
qwen_r_maxlen   = 8192

_qwen_prefix = (
    '<|im_start|>system\n'
    'Judge whether the Document meets the requirements based on the '
    'Query and the Instruct provided. '
    'Note that the answer can only be "yes" or "no".'
    '<|im_end|>\n<|im_start|>user\n'
)
_qwen_suffix       = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
qwen_r_prefix_toks = qwen_r_tok.encode(_qwen_prefix, add_special_tokens=False)
qwen_r_suffix_toks = qwen_r_tok.encode(_qwen_suffix, add_special_tokens=False)

_QWEN_INSTRUCTION = (
    'Given a hospital accreditation policy query, retrieve the document '
    'chunk that contains the relevant standard, intent, or measurable '
    'elements that answer the query.'
)


def rerank_qwen(query: str, retrieved: list) -> list:
    """
    Returns list of (doc, norm_score) sorted by norm_score descending.
    norm_score = log_softmax yes-probability — absolute 0-1.
    """
    docs  = [doc for doc, _ in retrieved]
    pairs = [
        f'<Instruct>: {_QWEN_INSTRUCTION}\n'
        f'<Query>: {query}\n'
        f'<Document>: {doc.page_content[:400]}'
        for doc in docs
    ]

    inputs = qwen_r_tok(
        pairs,
        padding=False,
        truncation='longest_first',
        return_attention_mask=False,
        max_length=qwen_r_maxlen - len(qwen_r_prefix_toks) - len(qwen_r_suffix_toks),
    )
    for i, ele in enumerate(inputs['input_ids']):
        inputs['input_ids'][i] = qwen_r_prefix_toks + ele + qwen_r_suffix_toks
    inputs = qwen_r_tok.pad(
        inputs, padding=True, return_tensors='pt', max_length=qwen_r_maxlen
    )
    for key in inputs:
        inputs[key] = inputs[key].to(qwen_r_model.device)

    with torch.no_grad():
        logits = qwen_r_model(**inputs).logits[:, -1, :]
        tv     = logits[:, qwen_r_true_id]
        fv     = logits[:, qwen_r_false_id]
        scores = torch.nn.functional.log_softmax(
            torch.stack([fv, tv], dim=1), dim=1
        )[:, 1].exp().tolist()

    combined = [(doc, round(score, 4)) for doc, score in zip(docs, scores)]
    return sorted(combined, key=lambda x: x[1], reverse=True)

print(f'Qwen reranker loaded: {QWEN_MODEL_ID}')

## 8. Load LLM — Qwen2.5-7B-Instruct



In [ ]:
# Prompts
SYSTEM_PROMPT_1 = ("""
You are an expert healthcare accreditation assistant specialized in the
Joint Commission International (JCI) 7th Edition Standards (special edition 
for Singapore Ministry of Health).

You will be given:
- A user question
- One retrieved text chunk from the JCI 7th Edition document.

YOUR PRIMARY RULE:
You must answer the question using ONLY the information explicitly stated
in the provided chunk text. Do not use external knowledge, assumptions,
training data, or prior familiarity with JCI standards.

GROUNDING & ACCURACY REQUIREMENTS:
- Base every statement directly on the retrieved chunk text.
- Do NOT paraphrase in a way that changes regulatory intent.
- Do NOT infer requirements that are not clearly stated in the text.

WHEN INFORMATION IS INSUFFICIENT:
- If the chunk text does not fully answer the question, respond with:
  “The provided JCI ESS excerpts do not contain sufficient information
   to answer this question.”
- Optionally state what type of information is missing (e.g., definition,
  requirement, responsibility), without guessing or filling gaps.

PROHIBITED BEHAVIORS:
- Do NOT hallucinate standards, elements of performance, or requirements.
- Do NOT cite editions, chapters, or clauses unless they appear in the text.
- Do NOT provide clinical, legal, or operational advice beyond what is
  explicitly stated.

STYLE & TONE:
- Use clear, professional, and neutral language suitable for healthcare
  governance, audits, and compliance discussions.
- Prefer concise, structured answers (bullet points or short paragraphs)
  when appropriate.
- Avoid conversational filler or speculative language.

OPTIONAL CITATION BEHAVIOR (if metadata is present):
- If chunk identifiers (e.g., standard code, section title) are provided,
  reference them explicitly.
- If no identifiers are present, do not invent them.

TRANSPARENCY RULE:
If there is ambiguity in the text, acknowledge it explicitly and explain
the ambiguity based only on the wording in the chunks.

Your role is to faithfully reflect the JCI 7th Edition ESS chunk text content—
nothing more, nothing less.
"""
)

In [ ]:
qwen_llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)
qwen_llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)


def build_qwen_prompt(query: str, chunks) -> list:
    """
    Formats one or more chunks and a query into the message list for Qwen2.5.
    `chunks` can be a single Document or a list of Documents.
    """
    if isinstance(chunks, Document):
        chunks = [chunks]

    context_parts = []
    for idx, chunk in enumerate(chunks, 1):
        context_parts.append(
            f'[Chunk {idx} | Standard: {chunk.metadata.get("standard_code", "?")} | '
            f'Chapter: {chunk.metadata.get("chapter", "?")}]\n'
            f'{chunk.page_content.strip()}'
        )
    context_str = '\n\n'.join(context_parts)

    user_content = (
        f'Context:\n'
        f'{context_str}\n\n'
        f'Question: {query}\n\n'
        f'Answer (cite the standard code where relevant):'
    )
    qwen_input = [
        {'role': 'system', 'content': SYSTEM_PROMPT_1},
        {'role': 'user',   'content': user_content},
    ]
    return qwen_input


def generate_qwen_response(messages: list) -> str:
    text = qwen_llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = qwen_llm_tokenizer([text], return_tensors="pt").to(qwen_llm_model.device)

    generated_ids = qwen_llm_model.generate(
        **model_inputs,
        max_new_tokens=LLM_MAX_TOKENS
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    qwen_response = qwen_llm_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return qwen_response


print('LLM functions ready.')

## 9. Load FAISS DBs & Run Generation

For each query:
1. Retrieve top-10 chunks via BGE-large FAISS
2. Rerank with CrossEncoder → Top-1 answer + Top-3 combined answer
3. Rerank with Qwen       → Top-1 answer + Top-3 combined answer
1 row per query = 25 rows total
Total LLM calls: 25 queries × 2 answer types × 2 rerankers = 100 calls

In [ ]:
# ── Load FAISS DBs ────────────────────────────────────────────────────────
print(f'Loading FAISS DBs: {EMBEDDING_MODEL_NAME}')

llm_embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_STRING,
    encode_kwargs={'normalize_embeddings': True},
)
llm_dbs = {}
for chapter in chapter_chunks.keys():
    chapter_path = os.path.join(DB_SAVE_DIR, EMBEDDING_MODEL_NAME, chapter)
    if os.path.exists(chapter_path):
        llm_dbs[chapter] = FAISS.load_local(
            chapter_path, llm_embeddings,
            allow_dangerous_deserialization=True,
        )
print(f'Loaded {len(llm_dbs)} chapter DBs.')


answer_rows = []

for i, item in enumerate(gold_standard, 1):
    query    = item['query']
    expected = item['expected_standard']
    print(f'[{i:02d}/{len(gold_standard)}] {expected}')

    retrieved = retrieve(query, llm_dbs)

    cross_ranked = rerank_cross(query, retrieved)
    qwen_ranked  = rerank_qwen(query, retrieved[:FINAL_K])

    # ── Cross: Top-1 answer ───────────────────────────────────────────
    cross_top1_doc,   cross_top1_score  = cross_ranked[0]
    cross_top3        = cross_ranked[:3]
    cross_top3_docs   = [doc for doc, _ in cross_top3]
    cross_top3_scores = [score for _, score in cross_top3]
    cross_top3_stds   = [doc.metadata.get('standard_code') for doc in cross_top3_docs]

    cross_answer_top1 = generate_qwen_response(build_qwen_prompt(query, cross_top1_doc))
    cross_answer_top3 = generate_qwen_response(build_qwen_prompt(query, cross_top3_docs))

    # ── Qwen: Top-1 answer ────────────────────────────────────────────
    qwen_top1_doc,   qwen_top1_score  = qwen_ranked[0]
    qwen_top3        = qwen_ranked[:3]
    qwen_top3_docs   = [doc for doc, _ in qwen_top3]
    qwen_top3_scores = [score for _, score in qwen_top3]
    qwen_top3_stds   = [doc.metadata.get('standard_code') for doc in qwen_top3_docs]

    qwen_answer_top1 = generate_qwen_response(build_qwen_prompt(query, qwen_top1_doc))
    qwen_answer_top3 = generate_qwen_response(build_qwen_prompt(query, qwen_top3_docs))

    answer_rows.append({
        'query':             query,
        'expected_standard': expected,

        # ── CrossEncoder ──────────────────────────────────────────────
        # Top-1
        'cross_top1_standard': cross_top1_doc.metadata.get('standard_code'),
        'cross_top1_score':    cross_top1_score,
        'cross_top1_chunk':    cross_top1_doc.page_content.strip(),
        'cross_answer_top1':   cross_answer_top1,
        # Top-3
        'cross_top3_standards': str(cross_top3_stds),   # e.g. "['ACC.1', 'ACC.2', 'ACC.3']"
        'cross_top3_scores':    str(cross_top3_scores),
        'cross_top3_chunks':    '\n\n---\n\n'.join(d.page_content.strip() for d in cross_top3_docs),
        'cross_answer_top3':    cross_answer_top3,

        # ── Qwen Reranker ─────────────────────────────────────────────
        # Top-1
        'qwen_top1_standard': qwen_top1_doc.metadata.get('standard_code'),
        'qwen_top1_score':    qwen_top1_score,
        'qwen_top1_chunk':    qwen_top1_doc.page_content.strip(),
        'qwen_answer_top1':   qwen_answer_top1,
        # Top-3
        'qwen_top3_standards': str(qwen_top3_stds),
        'qwen_top3_scores':    str(qwen_top3_scores),
        'qwen_top3_chunks':    '\n\n---\n\n'.join(d.page_content.strip() for d in qwen_top3_docs),
        'qwen_answer_top3':    qwen_answer_top3,
    })

# ── Save ──────────────────────────────────────────────────────────────────
answers_df = pd.DataFrame(answer_rows)

cross_cols = [
    'query', 'expected_standard',
    'cross_top1_standard', 'cross_top1_score', 'cross_top1_chunk', 'cross_answer_top1',
    'cross_top3_standards', 'cross_top3_scores', 'cross_top3_chunks', 'cross_answer_top3',
]
answers_df[cross_cols].to_csv('QWENL_CROSS_ANSWERS_prompt1.csv', index=False)
print(f'Saved: QWENL_CROSS_ANSWERS_prompt1.csv  ({len(answers_df)} rows)')

qwen_cols = [
    'query', 'expected_standard',
    'qwen_top1_standard', 'qwen_top1_score', 'qwen_top1_chunk', 'qwen_answer_top1',
    'qwen_top3_standards', 'qwen_top3_scores', 'qwen_top3_chunks', 'qwen_answer_top3',
]
answers_df[qwen_cols].to_csv('QWEN_QWEN_ANSWERS_Prompt1.csv', index=False)
print(f'Saved: QWEN_QWEN_ANSWERS_Prompt1.csv  ({len(answers_df)} rows)')

## 10. Summary

In [ ]:
df = pd.DataFrame(answer_rows)

print('RERANKER RETRIEVAL ACCURACY SUMMARY')
print('(% of queries where retrieved standard matches expected standard)')
print('='*60)

for reranker, top1_col, top3_col in [
    ('CrossEncoder', 'cross_top1_standard', 'cross_top3_standards'),
    ('Qwen',         'qwen_top1_standard',  'qwen_top3_standards'),
]:
    top1_acc = (df[top1_col] == df['expected_standard']).mean() * 100

    top3_acc = df.apply(
        lambda row: row['expected_standard'] in row[top3_col], axis=1
    ).mean() * 100

    print(f'\n{reranker}:')
    print(f'  Top-1 accuracy (exact match):          {top1_acc:.1f}%')
    print(f'  Top-3 accuracy (expected in top-3):    {top3_acc:.1f}%')

print()

print('RERANKER SCORE SUMMARY')
print('='*60)
score_summary = df.agg(
    cross_top1_avg_score = ('cross_top1_score', 'mean'),
    cross_top3_avg_score = ('cross_top3_scores', lambda x:
                            x.apply(lambda s: np.mean(eval(s))).mean()),
    qwen_top1_avg_score  = ('qwen_top1_score', 'mean'),
    qwen_top3_avg_score  = ('qwen_top3_scores', lambda x:
                            x.apply(lambda s: np.mean(eval(s))).mean()),
).round(3)
print(score_summary.to_string())
print()
print('If a reranker gives high scores but low accuracy it is confidently wrong.')